In [ ]:
##################################################
# --- Inserire il modello e l'url per lo SCRAPING
##################################################

modello_per_analisi = 'Audi_A3'  
# Opzioni possibili: Mercedes_Classe_A, BMW_Serie_1, Audi_A3, Fiat_Panda

# Lista degli URL (multipli in caso di AutoScout, in quanto mostra solo 400 annunci a volta)
urls = [
    'https://www.autoscout24.it/lst/audi/a3?atype=C&custtype=D&cy=I&damaged_listing=exclude&desc=0&fregfrom=2020&powertype=kw&pricefrom=2000&priceto=25000&search_id=12ielxe9rq4&sort=standard&source=homepage_search-mask&ustate=N%2CU',
    'https://www.autoscout24.it/lst/audi/a3?atype=C&custtype=D&cy=I&damaged_listing=exclude&desc=0&fregfrom=2020&powertype=kw&pricefrom=25000&priceto=30000&search_id=1udnb3io5rt&sort=standard&source=homepage_search-mask&ustate=N%2CU',
    'https://www.autoscout24.it/lst/audi/a3?atype=C&custtype=D&cy=I&damaged_listing=exclude&desc=0&fregfrom=2020&kmto=20000&powertype=kw&pricefrom=30000&priceto=40000&search_id=127dygmh87y&sort=standard&source=homepage_search-mask&ustate=N%2CU',
    'https://www.autoscout24.it/lst/audi/a3?atype=C&custtype=D&cy=I&damaged_listing=exclude&desc=0&fregfrom=2020&kmfrom=20000&powertype=kw&pricefrom=30000&priceto=40000&search_id=18hpxf5w08w&sort=standard&source=homepage_search-mask&ustate=N%2CU',
    'https://www.autoscout24.it/lst/audi/a3?atype=C&custtype=D&cy=I&damaged_listing=exclude&desc=0&fregfrom=2020&powertype=kw&pricefrom=40000&search_id=swdhtz0w09&sort=standard&source=homepage_search-mask&ustate=N%2CU'
]

### 0. Import funzioni e librerie

In [ ]:
import os
import re
import time
import random
import requests
import pandas as pd
import sys
from pathlib import Path
from bs4 import BeautifulSoup
from Scraping_Functions import autoscout_scraper

In [ ]:
# Percorso della root del progetto (1 livelli sopra)
project_root = Path.cwd().parents[0]

# Percorsi delle cartelle contenenti i moduli
output_folder = project_root / 'Materiali' / 'Dataset_grezzi' / modello_per_analisi

# Aggiungi al sys.path
sys.path.extend(str(output_folder))

#Verifica se la cartella esiste, se no la crea
output_folder.mkdir(parents=True, exist_ok=True)

### 1. Scraping - Autoscout

In [ ]:
# Lista per raccogliere tutti i DataFrame
all_dataframes = []

# Ciclo su tutti gli URL
for i, url in enumerate(urls, start=1):
    print(f"\n🔍 Scraping URL {i}/{len(urls)}...")
    df_temp = autoscout_scraper(url_template=url + "&page={}", max_pages=20)
    if df_temp is not None and not df_temp.empty:
        df_temp['Categoria'] = f'URL_{i}'  #colonna per sapere da quale URL proviene
        all_dataframes.append(df_temp)
    else:
        print(f"⚠ Nessun dato trovato per URL {i}")

# Concatenazione dei tutti i DataFrame in uno solo
if all_dataframes:
    df_final = pd.concat(all_dataframes, ignore_index=True)
    print("\n✅ Raccolti tutti i dati!")
    print(df_final.head())
else:
    print("❌ Nessun dato trovato da nessun URL.")

### 2. Export

In [ ]:
print(df_final.info())
df_final.head()

In [ ]:
# Salvataggio del file
output_file = output_folder / "data_autoscout.csv"
df = df_final.drop(columns='Categoria', errors='ignore')
df.to_csv(output_file, index=False)
print(f"""💾 File salvato, del modello {modello_per_analisi},
   nel percorso: {output_file}""")